# Graph Programming — 04: Island Problems

Island problems are **grid DFS/BFS** problems. They're extremely common in interviews.

The mental model:  
- Grid cells = nodes  
- Adjacent land cells = edges  
- An "island" = a connected component of `'1'` cells

## The Core Template

```python
def dfs(r, c):
    if out_of_bounds or not_land or already_visited:
        return
    mark_visited
    dfs(r+1, c); dfs(r-1, c); dfs(r, c+1); dfs(r, c-1)
```

---

## Problem 1: Number of Islands (LC 200) ⭐ Classic

In [ ]:
# LC 200 — Number of Islands
# Given a 2D grid of '1' (land) and '0' (water), count the number of islands.
# An island is surrounded by water and formed by connecting adjacent lands horizontally/vertically.

def numIslands(grid):
    if not grid:
        return 0

    rows, cols = len(grid), len(grid[0])
    count = 0

    def dfs(r, c):
        # Stop if: out of bounds OR water OR already visited
        if r < 0 or r >= rows or c < 0 or c >= cols or grid[r][c] != '1':
            return
        grid[r][c] = '#'   # sink/mark visited
        dfs(r+1, c)
        dfs(r-1, c)
        dfs(r, c+1)
        dfs(r, c-1)

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1':   # found new island
                count += 1
                dfs(r, c)           # sink the whole island

    return count

grid1 = [
    ['1','1','1','1','0'],
    ['1','1','0','1','0'],
    ['1','1','0','0','0'],
    ['0','0','0','0','0']
]
grid2 = [
    ['1','1','0','0','0'],
    ['1','1','0','0','0'],
    ['0','0','1','0','0'],
    ['0','0','0','1','1']
]

print(numIslands(grid1))  # 1
print(numIslands(grid2))  # 3

---
## Problem 2: Max Area of Island (LC 695)

Same as Number of Islands, but return the **area** (cell count) of the largest island.

In [ ]:
# LC 695 — Max Area of Island
# DFS returns how many cells it visited (the area of this island)

def maxAreaOfIsland(grid):
    rows, cols = len(grid), len(grid[0])

    def dfs(r, c):
        if r < 0 or r >= rows or c < 0 or c >= cols or grid[r][c] != 1:
            return 0
        grid[r][c] = 0   # mark visited
        # 1 (this cell) + area of all connected land
        return 1 + dfs(r+1,c) + dfs(r-1,c) + dfs(r,c+1) + dfs(r,c-1)

    max_area = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1:
                max_area = max(max_area, dfs(r, c))

    return max_area

grid = [
    [0,0,1,0,0,0,0,1,0,0,0,0,0],
    [0,0,0,0,0,0,0,1,1,1,0,0,0],
    [0,1,1,0,1,0,0,0,0,0,0,0,0],
    [0,1,0,0,1,1,0,0,1,0,1,0,0],
    [0,1,0,0,1,1,0,0,1,1,1,0,0],
    [0,0,0,0,0,0,0,0,0,0,1,0,0],
    [0,0,0,0,0,0,0,1,1,1,0,0,0],
    [0,0,0,0,0,0,0,1,1,0,0,0,0]
]
print(maxAreaOfIsland(grid))  # 6

---
## Problem 3: Surrounded Regions (LC 130)

Flip all `'O'` regions that are completely surrounded by `'X'` to `'X'`.  
Regions touching the border are NOT surrounded.

**Reverse thinking trick:**  
Instead of finding surrounded Os, find SAFE Os (those connected to the border).  
Mark them, then flip everything else.

```
X X X X       X X X X
X O O X  -->  X X X X
X X O X       X X X X
X O X X       X O X X   <- border O stays
```

In [ ]:
# LC 130 — Surrounded Regions

def solve(board):
    if not board:
        return
    rows, cols = len(board), len(board[0])

    def dfs(r, c):
        # Mark safe O's (connected to border) with 'S'
        if r < 0 or r >= rows or c < 0 or c >= cols or board[r][c] != 'O':
            return
        board[r][c] = 'S'   # safe
        dfs(r+1,c); dfs(r-1,c); dfs(r,c+1); dfs(r,c-1)

    # Step 1: Mark all O's connected to the border as 'S' (safe)
    for r in range(rows):
        for c in range(cols):
            if (r == 0 or r == rows-1 or c == 0 or c == cols-1) and board[r][c] == 'O':
                dfs(r, c)

    # Step 2: Flip remaining O's to X, restore S's back to O
    for r in range(rows):
        for c in range(cols):
            if board[r][c] == 'O':
                board[r][c] = 'X'   # surrounded — flip
            elif board[r][c] == 'S':
                board[r][c] = 'O'   # safe — restore

board = [
    ['X','X','X','X'],
    ['X','O','O','X'],
    ['X','X','O','X'],
    ['X','O','X','X']
]
solve(board)
for row in board:
    print(row)
# Row 4 'O' stays (it's on the border)

---
## Problem 4: Pacific Atlantic Water Flow (LC 417)

Given a height matrix, water flows from higher to lower (or equal) cells.  
Find cells where water can flow to **both** the Pacific (top/left) and Atlantic (bottom/right) oceans.

**Reverse flow trick:** Instead of flowing down from each cell, flow UP from the oceans.  
Find all cells reachable from Pacific edge and all cells reachable from Atlantic edge.  
Answer = intersection.

In [ ]:
# LC 417 — Pacific Atlantic Water Flow

def pacificAtlantic(heights):
    rows, cols = len(heights), len(heights[0])
    DIRS = [(0,1),(0,-1),(1,0),(-1,0)]

    def bfs(starts):
        """BFS from ocean border inward (reverse flow)."""
        from collections import deque
        visited = set(starts)
        queue = deque(starts)
        while queue:
            r, c = queue.popleft()
            for dr, dc in DIRS:
                nr, nc = r+dr, c+dc
                if (0 <= nr < rows and 0 <= nc < cols
                        and (nr, nc) not in visited
                        and heights[nr][nc] >= heights[r][c]):  # water flows DOWN so we go UP
                    visited.add((nr, nc))
                    queue.append((nr, nc))
        return visited

    # Pacific touches top row + left col
    pacific_starts = [(r, 0) for r in range(rows)] + [(0, c) for c in range(cols)]
    # Atlantic touches bottom row + right col
    atlantic_starts = [(r, cols-1) for r in range(rows)] + [(rows-1, c) for c in range(cols)]

    pacific = bfs(pacific_starts)
    atlantic = bfs(atlantic_starts)

    return sorted(pacific & atlantic)  # intersection

heights = [
    [1,2,2,3,5],
    [3,2,3,4,4],
    [2,4,5,3,1],
    [6,7,1,4,5],
    [5,1,1,2,4]
]
print(pacificAtlantic(heights))
# [(0,4),(1,3),(1,4),(2,2),(3,0),(3,1),(4,0)]

---
## Problem 5: Number of Islands — BFS Version

You can solve the same problem with BFS. Useful to know both approaches.

In [ ]:
from collections import deque

def numIslands_BFS(grid):
    if not grid:
        return 0
    rows, cols = len(grid), len(grid[0])
    count = 0
    DIRS = [(0,1),(0,-1),(1,0),(-1,0)]

    def bfs(r, c):
        queue = deque([(r, c)])
        grid[r][c] = '#'
        while queue:
            row, col = queue.popleft()
            for dr, dc in DIRS:
                nr, nc = row+dr, col+dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == '1':
                    grid[nr][nc] = '#'
                    queue.append((nr, nc))

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1':
                count += 1
                bfs(r, c)
    return count

grid = [
    ['1','1','0'],
    ['0','1','0'],
    ['0','0','1']
]
print(numIslands_BFS(grid))  # 2

---
## Island Problems — Pattern Summary

| Problem | Technique | Key Insight |
|---|---|---|
| Number of Islands | DFS/BFS + sink | Count DFS starts |
| Max Area | DFS returns count | Accumulate cell count in DFS |
| Surrounded Regions | Reverse DFS from border | Mark safe first, flip rest |
| Pacific Atlantic | Reverse BFS from both oceans | Intersect reachable sets |
| Rotting Oranges | Multi-source BFS | Seed all sources at once |

**Common trick:** When you need to avoid revisiting, either:
- Modify the grid in place (`'1'` → `'#'`)
- Keep a `visited = set()` of `(r, c)` tuples

**Next:** `05_course_schedule.ipynb` — Topological Sort & Cycle Detection